# Explainer Evaluation Notebook

Evaluates `generate_explanation` across all 6 intent types using two independent judges:
- **Claude Sonnet** (`claude-sonnet-4-6`) — Anthropic API
- **Groq Llama** (`llama-3.3-70b-versatile`) — Groq API

### Evaluation pipeline
1. Run the full pipeline (classify → route_and_execute → generate_explanation) for each intent
2. **Factual grounding check** — programmatic: do key values from the context appear in the response?
3. **LLM-as-judge** — rubric-based scoring (0–5) on 4 dimensions
4. **Judge agreement** — compare Claude vs Groq scores
5. **Flag low scorers** — surface responses needing improvement

In [ ]:
import os, sys, json, re
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import anthropic
from groq import Groq

from agent_tools.workflow_tools import classify_intent, route_and_execute, Intent
from agent_tools.workflow_tools.agent_llm import KeyRotator

_gemini = KeyRotator()
_claude = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY", ""))
_groq   = Groq(api_key=os.environ.get("GROQ_API_KEY", ""))

CLAUDE_MODEL = "claude-sonnet-4-6"
GROQ_MODEL   = "llama-3.3-70b-versatile"

print("Clients initialised.")

---
## Section 1: Generate Explanations
Run the full pipeline for each intent. The follow-up test is chained off the real full_analysis response.

In [ ]:
PORTFOLIO = {
    "tickers": ["AAPL", "GOOGL", "MSFT"],
    "weights": [0.4, 0.3, 0.3],
    "investment_amount": 10000,
    "currency": "USD",
}

TEST_CASES = [
    {
        "id": "full_analysis",
        "query": "Give me a full risk analysis of my portfolio",
        "portfolio_changed": True,
        "history": [],
        "cache": {},
    },
    {
        "id": "specific_metric",
        "query": "Show me my Sharpe ratio and max drawdown",
        "portfolio_changed": True,
        "history": [],
        "cache": {},
    },
    {
        "id": "concept_explanation",
        "query": "What is Value at Risk and why does it matter?",
        "portfolio_changed": False,
        "history": [],
        "cache": {},
    },
    {
        "id": "trend_prediction",
        "query": "What does the volatility forecast look like for my portfolio?",
        "portfolio_changed": True,
        "history": [],
        "cache": {},
    },
    {
        "id": "general_chat",
        "query": "Hi! What can you help me with?",
        "portfolio_changed": False,
        "history": [],
        "cache": {},
    },
    # follow_up is added below after full_analysis runs
]

print(f"{len(TEST_CASES)} base test cases defined.")

In [ ]:
def run_pipeline(query, history, portfolio_changed, cache):
    intent = _gemini.call_with_retry(
        lambda c: classify_intent(
            message=query,
            recent_history=history,
            portfolio_changed=portfolio_changed,
            client=c,
        )
    )
    workflow = route_and_execute(
        intent,
        user_query=query,
        portfolio=PORTFOLIO,
        is_first_portfolio=not bool(cache),
        portfolio_changed=portfolio_changed,
        recent_history=history,
        cache=cache,
    )
    return intent, workflow


results = {}

for tc in TEST_CASES:
    print(f"Running {tc['id']}...", end=" ", flush=True)
    intent, workflow = run_pipeline(
        tc["query"], tc["history"], tc["portfolio_changed"], tc["cache"]
    )
    all_metrics = (workflow.cache.get("metrics") or {}).get("all_metrics")
    results[tc["id"]] = {
        "query":    tc["query"],
        "intent":   intent.primary_intent.value,
        "response": workflow.content,
        "cache":    workflow.cache,
        "context": {
            "metrics":        all_metrics,
            "risk_level":     workflow.cache.get("risk_level"),
            "trend_forecast": workflow.cache.get("trend_forecast"),
        },
    }
    print(f"{len(workflow.content)} chars  [{intent.primary_intent.value}]")

# Chain follow_up off the real full_analysis response
full_cache    = results["full_analysis"]["cache"]
full_response = results["full_analysis"]["response"]
follow_history = [
    {"role": "user",      "content": "Give me a full risk analysis of my portfolio"},
    {"role": "assistant", "content": full_response},
    {"role": "user",      "content": "Can you say more about what you just told me?"},
]
print("Running follow_up...", end=" ", flush=True)
intent, workflow = run_pipeline(
    "Can you say more about what you just told me?",
    follow_history, False, full_cache,
)
all_metrics = (full_cache.get("metrics") or {}).get("all_metrics")
results["follow_up"] = {
    "query":    "Can you say more about what you just told me?",
    "intent":   intent.primary_intent.value,
    "response": workflow.content,
    "cache":    workflow.cache,
    "context": {
        "metrics":      all_metrics,
        "risk_level":   full_cache.get("risk_level"),
        "chat_history": follow_history,
    },
}
print(f"{len(workflow.content)} chars  [{intent.primary_intent.value}]")
print(f"\nDone — {len(results)} explanations generated.")

In [ ]:
# Preview all responses
for tid, r in results.items():
    print(f"\n{'='*60}")
    print(f"[{tid}]  intent classified as: {r['intent']}")
    print(f"{'='*60}")
    print(r["response"][:500].strip(), "..." if len(r["response"]) > 500 else "")

---
## Section 2: Factual Grounding Check
Programmatic check — do the key values from the context dict appear in the response?
No LLM needed. Checks: risk level label, key metric values, forecast direction, prior conversation reference.

In [ ]:
def grounding_check(test_id: str, r: dict) -> dict:
    """Check that key facts from the context appear in the response."""
    text = r["response"].lower()
    ctx  = r["context"]
    checks = {}

    # Risk level label
    if ctx.get("risk_level"):
        label = ctx["risk_level"]["label"].lower()
        checks["risk_label"] = label in text

    # Key metric values — check both decimal and percentage forms
    if ctx.get("metrics"):
        for key in ["portfolio_volatility", "sharpe_ratio", "max_drawdown", "var_95"]:
            val = ctx["metrics"].get(key)
            if val is None:
                continue
            dec = f"{val:.2f}"           # e.g.  0.24
            pct = f"{abs(val)*100:.0f}"  # e.g.  24
            checks[f"{key}_cited"] = (dec in text or pct in text)

    # Forecast direction
    if ctx.get("trend_forecast"):
        direction = ctx["trend_forecast"]["predicted_direction"].lower()
        checks["forecast_direction"] = direction in text

    # Follow-up: references something from the prior assistant message
    if test_id == "follow_up" and ctx.get("chat_history"):
        prior_words = [
            w for w in ctx["chat_history"][1]["content"].lower().split()
            if len(w) > 5
        ][:20]
        checks["references_prior"] = any(w in text for w in prior_words)

    passed = sum(1 for v in checks.values() if v)
    total  = len(checks)
    return {
        "checks": checks,
        "passed": passed,
        "total":  total,
        "score":  passed / total if total else 1.0,
    }

grounding = {tid: grounding_check(tid, r) for tid, r in results.items()}
print("Grounding checks complete.")

In [ ]:
rows = []
for tid, g in grounding.items():
    row = {
        "test_case": tid,
        "passed":    g["passed"],
        "total":     g["total"],
        "score":     f"{g['score']:.0%}",
    }
    row.update({k: "✓" if v else "✗" for k, v in g["checks"].items()})
    rows.append(row)

df_grounding = pd.DataFrame(rows).set_index("test_case")
df_grounding

---
## Section 3: LLM-as-Judge
Rubric-based scoring (0–5) on 4 dimensions, run by both Claude Sonnet and Groq Llama independently.

In [ ]:
DIMENSIONS = ["factual_accuracy", "intent_alignment", "completeness", "clarity"]

RUBRIC = """
Score the chatbot response on these 4 dimensions (0-5 each):

1. factual_accuracy
   5 = all numbers and labels exactly match the provided key facts
   3 = mostly correct with minor imprecision
   0 = hallucinated or clearly wrong values

2. intent_alignment
   5 = directly and fully answers what the user asked, stays on topic
   3 = partially addresses the question or includes unnecessary tangents
   0 = ignores the question or goes completely off-topic

3. completeness  (intent-specific expectations)
   full_analysis    : covers risk level + key drivers + notable metrics + outlook
   specific_metric  : covers all requested metrics with values + benchmark comparison
   concept_explanation: clear definition + why it matters + illustrates with portfolio numbers
   trend_prediction : forecast values + factors that could invalidate it + uncertainty caveats
   follow_up        : directly addresses the specific follow-up + references prior context
   general_chat     : friendly tone + mentions key capabilities
   Scale: 5=fully complete, 3=mostly complete, 0=missing key elements

4. clarity
   5 = clear, well-structured, technical terms explained, easy to follow
   3 = readable but somewhat dense, disorganised, or missing explanations
   0 = confusing, poorly structured, or jargon-heavy without explanation

Return ONLY valid JSON with no extra text:
{"factual_accuracy": X, "intent_alignment": X, "completeness": X, "clarity": X, "reasoning": "1-2 sentences"}
"""

print("Rubric defined.")

In [ ]:
def extract_key_facts(r: dict) -> str:
    """Summarise the key facts the chatbot had access to."""
    lines = [f"Intent: {r['intent']}", f"User query: {r['query']}"]
    ctx = r["context"]
    if ctx.get("risk_level"):
        rl = ctx["risk_level"]
        lines.append(f"Risk level: {rl['label']} (confidence: {rl['confidence']:.2f})")
    if ctx.get("metrics"):
        m = ctx["metrics"]
        for k in ["portfolio_volatility", "sharpe_ratio", "max_drawdown",
                  "var_95", "beta", "sortino_ratio"]:
            if k in m:
                lines.append(f"{k}: {m[k]:.4f}")
    if ctx.get("trend_forecast"):
        tf = ctx["trend_forecast"]
        lines.append(
            f"Forecast: direction={tf['predicted_direction']}, "
            f"vol={tf['predicted_volatility']:.4f}, "
            f"confidence={tf['confidence']:.2f}"
        )
    if ctx.get("chat_history"):
        prior = ctx["chat_history"][1]["content"][:200]
        lines.append(f"Prior assistant response (first 200 chars): {prior}")
    return "\n".join(lines)


def build_judge_prompt(r: dict) -> str:
    facts    = extract_key_facts(r)
    response = r["response"][:2000]  # cap to control token usage
    return (
        f"Key facts the chatbot had access to:\n{facts}\n\n"
        f"Chatbot response:\n{response}\n\n"
        f"{RUBRIC}"
    )


def parse_scores(text: str) -> dict:
    """Extract JSON scores, handling markdown code fences."""
    text = re.sub(r"```json|```", "", text).strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return json.loads(match.group())
    raise ValueError(f"No JSON found in judge response: {text[:300]}")


def judge_claude(r: dict) -> dict:
    msg = _claude.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=400,
        messages=[{"role": "user", "content": build_judge_prompt(r)}],
    )
    return parse_scores(msg.content[0].text)


def judge_groq(r: dict) -> dict:
    msg = _groq.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": build_judge_prompt(r)}],
        max_tokens=400,
        temperature=0.0,
    )
    return parse_scores(msg.choices[0].message.content)


print("Judge functions defined.")

In [ ]:
judge_results = {}

for tid, r in results.items():
    print(f"Judging [{tid}]...", end="  ", flush=True)
    claude_scores, groq_scores = {}, {}

    try:
        claude_scores = judge_claude(r)
        print("claude ✓", end="  ", flush=True)
    except Exception as e:
        claude_scores = {d: None for d in DIMENSIONS}
        claude_scores["reasoning"] = f"ERROR: {e}"
        print(f"claude ✗  ", end="", flush=True)

    try:
        groq_scores = judge_groq(r)
        print("groq ✓")
    except Exception as e:
        groq_scores = {d: None for d in DIMENSIONS}
        groq_scores["reasoning"] = f"ERROR: {e}"
        print(f"groq ✗")

    judge_results[tid] = {"claude": claude_scores, "groq": groq_scores}

print("\nAll judgements complete.")

---
## Section 4: Results

In [ ]:
# Scores table: one row per (test_case, judge)
rows = []
for tid, jr in judge_results.items():
    for judge_name, scores in jr.items():
        row = {"test_case": tid, "judge": judge_name}
        for d in DIMENSIONS:
            row[d] = scores.get(d)
        valid = [scores.get(d) for d in DIMENSIONS if scores.get(d) is not None]
        row["avg"] = round(sum(valid) / len(valid), 2) if valid else None
        rows.append(row)

df_scores = pd.DataFrame(rows)

# Pivot: rows = test_case, columns = (dimension, judge)
df_pivot = df_scores.pivot_table(
    index="test_case",
    columns="judge",
    values=DIMENSIONS + ["avg"],
).round(2)

print("Scores (0–5 per dimension, 5 = best)\n")
df_pivot

In [ ]:
# Judge reasoning for each test case
print("Judge reasoning:\n")
for tid, jr in judge_results.items():
    print(f"[{tid}]")
    for judge_name, scores in jr.items():
        print(f"  {judge_name}: {scores.get('reasoning', 'n/a')}")
    print()

In [ ]:
# Judge agreement: how closely do Claude and Groq agree?
agree_rows = []
for tid in results:
    c = judge_results[tid]["claude"]
    g = judge_results[tid]["groq"]
    for d in DIMENSIONS:
        cv, gv = c.get(d), g.get(d)
        if cv is not None and gv is not None:
            agree_rows.append({
                "test_case": tid,
                "dimension": d,
                "claude":    cv,
                "groq":      gv,
                "diff":      abs(cv - gv),
                "agree":     abs(cv - gv) <= 1,
            })

df_agree = pd.DataFrame(agree_rows)

print("Agreement by dimension (diff ≤ 1 counts as agree):")
display(
    df_agree.groupby("dimension")[["agree", "diff"]]
    .agg({"agree": "mean", "diff": "mean"})
    .rename(columns={"agree": "agree_rate", "diff": "mean_diff"})
    .round(2)
)
print(f"\nOverall agreement rate: {df_agree['agree'].mean():.0%}")
print(f"Overall mean score diff: {df_agree['diff'].mean():.2f} points")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, judge_name in zip(axes, ["claude", "groq"]):
    data = []
    test_ids = list(results.keys())
    for tid in test_ids:
        scores = judge_results[tid][judge_name]
        data.append([scores.get(d) or 0 for d in DIMENSIONS])

    mat = np.array(data, dtype=float)
    im  = ax.imshow(mat, vmin=0, vmax=5, cmap="RdYlGn", aspect="auto")

    ax.set_xticks(range(len(DIMENSIONS)))
    ax.set_xticklabels([d.replace("_", "\n") for d in DIMENSIONS], fontsize=9)
    ax.set_yticks(range(len(test_ids)))
    ax.set_yticklabels(test_ids, fontsize=9)
    ax.set_title(f"{judge_name.capitalize()} ({CLAUDE_MODEL if judge_name == 'claude' else GROQ_MODEL})",
                 fontsize=10, fontweight="bold")

    for i in range(len(test_ids)):
        for j in range(len(DIMENSIONS)):
            val = mat[i, j]
            ax.text(j, i, f"{val:.0f}", ha="center", va="center",
                    fontsize=11, fontweight="bold",
                    color="white" if val < 2 or val > 4 else "black")

fig.colorbar(im, ax=axes, label="Score (0–5)", shrink=0.8)
fig.suptitle("Explainer Evaluation — LLM-as-Judge Scores", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("docs/eval_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Heatmap saved to docs/eval_heatmap.png")

In [ ]:
LOW_THRESHOLD = 3

print(f"=== Low scores (any dimension < {LOW_THRESHOLD}) ===\n")
flagged = False

for tid, jr in judge_results.items():
    for judge_name, scores in jr.items():
        low = {d: scores.get(d) for d in DIMENSIONS if (scores.get(d) or 5) < LOW_THRESHOLD}
        if low:
            flagged = True
            print(f"  [{judge_name.upper()}] {tid}")
            for d, v in low.items():
                print(f"    {d}: {v}/5")
            print(f"    → {scores.get('reasoning', '')}\n")

if not flagged:
    print("  No low scores — all dimensions ≥ 3 across both judges.")

print("\n=== Per-judge average scores ===")
for judge_name in ["claude", "groq"]:
    avgs = {}
    for d in DIMENSIONS:
        vals = [judge_results[tid][judge_name].get(d) for tid in results
                if judge_results[tid][judge_name].get(d) is not None]
        avgs[d] = round(sum(vals)/len(vals), 2) if vals else None
    overall = round(sum(v for v in avgs.values() if v) / len(DIMENSIONS), 2)
    print(f"  {judge_name}: {avgs}  →  avg={overall}")